In [28]:
import sys, random, importlib
sys.path.insert(0, "..")
import torch

# Reload to pick up any in-session changes to sorl_trainer
import sorl.sorl_trainer as _st; importlib.reload(_st)
from sorl.sorl_trainer import sorl_search, infer_insert_mask, insert_tokens_with_padding
from sorl.trainer_ablate import _drop_nl_prefix_m_set
from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper
from data.pt_dataset import get_dataset, collate_fn

# ── Config ──────────────────────────────────────────────────────────────
MODEL_NAME  = "Qwen/Qwen3-0.6B"
ABS_VOCAB   = 32
K           = 4
N_SAMPLES   = 4
ANSWER_TOK  = 820   # "####" delimiter in GSM8K

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = SorlModelWrapper.from_pretrained(MODEL_NAME, abstract_vocab_size_list=[ABS_VOCAB])
model     = model.to(device).eval()

base_vocab = int(model.vocab_sizes[0].item())
pad_id     = tokenizer.pad_token_id

# ── GSM8K batch ──────────────────────────────────────────────────────────
ds         = get_dataset("gsm8k", split="train", tokenizer=tokenizer, max_length=256)
batch      = collate_fn([ds[i] for i in range(N_SAMPLES)])
input_ids  = batch["input_ids"].to(device)
attn_mask  = batch["attention_mask"].to(device)
prompt_len = batch["prompt_len"].to(device)

# ── Decode helper ─────────────────────────────────────────────────────────
def decode_annotated(ids_1d, valid_len=None):
    """NL tokens → text, abstract tokens → [ABS]. Pass a 1-D id tensor."""
    n = valid_len if valid_len is not None else len(ids_1d)
    parts, buf = [], []
    for tid in ids_1d[:n].tolist():
        if tid >= base_vocab:
            if buf: parts.append(tokenizer.decode(buf, skip_special_tokens=False)); buf = []
            parts.append("[ABS]")
        else:
            buf.append(tid)
    if buf: parts.append(tokenizer.decode(buf, skip_special_tokens=False))
    return "".join(parts)

print(f"device={device}  base_vocab={base_vocab}  abs_vocab={ABS_VOCAB}  K={K}")
print(f"GSM8K lens: {[int(attn_mask[b].sum()) for b in range(N_SAMPLES)]}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


device=cpu  base_vocab=151936  abs_vocab=32  K=4
GSM8K lens: [100, 99, 164, 183]


In [ ]:
# ── Design: Prefix-First Insertion Mask (closes the loop) ──────────────────
#
# FLAWED approach (move_abs_to_cot_prefix):
#   1. infer_insert_mask places ABS slots interleaved: [Q][C0 ABS C1 C2 ABS C3 ...][#### ans]
#   2. recursion() fills ABS via Jacobi in this interleaved layout
#   3. move_abs_to_cot_prefix rearranges post-hoc: [Q][ABS×N][CoT][#### ans]
#   4. Train on the rearranged layout
#   Problem: the model searches in interleaved format but trains on prefix format.
#   → The model's search ability (Jacobi decoding on the prefix) NEVER improves,
#     because search always runs on the interleaved layout.
#   → The ABS targets in the prefix are not increasingly meaningful over training.
#
# CORRECT approach (prefix-first insertion):
#   1. Directly construct [Q][ABS_PLACEHOLDER×N][CoT][#### ans]
#   2. recursion() fills the contiguous ABS prefix via Jacobi
#   3. Train on the SAME layout that search operates on
#   → Training improves model's ability to fill the prefix ABS block
#   → Next search uses the improved model → better ABS values
#   → Loop is closed: train ↔ search on identical layout
#
# Implementation: replace infer_insert_mask + insert_tokens_with_padding
# with insert_prefix_abs() that directly builds the prefix layout.


In [32]:
# Tricks: 
# 1. Randomize count of ABS_MASK
# 2. NL replacement. 
# 3. NL dropping

In [33]:
# ── Kernel 1: Prefix-first insertion + Jacobi refinement ──────────────────
#
# insert_prefix_abs() directly constructs:
#   [Q₁...Qₚ] [PLACEHOLDER×N] [C₁...Cₘ] [#### ans] [PAD...]
#
# Then recursion() fills the N placeholder slots via Jacobi decoding.
# Search and training operate on the SAME layout → loop is closed.

def insert_prefix_abs(data, attn, prompt_len, n_abs, placeholder_token, pad_token_id):
    """Insert n_abs contiguous ABS placeholders at the start of each response."""
    B, L = data.shape
    new_L = L + n_abs
    new_data = torch.full((B, new_L), pad_token_id, device=data.device, dtype=data.dtype)
    new_attn = torch.zeros((B, new_L), device=data.device, dtype=attn.dtype)
    for b in range(B):
        pl = prompt_len[b].item()
        valid = int(attn[b].sum().item())
        # prompt (unchanged)
        new_data[b, :pl] = data[b, :pl]
        new_attn[b, :pl] = 1
        # ABS prefix block
        new_data[b, pl:pl+n_abs] = placeholder_token
        new_attn[b, pl:pl+n_abs] = 1
        # response (CoT + answer)
        resp_len = valid - pl
        new_data[b, pl+n_abs:pl+n_abs+resp_len] = data[b, pl:valid]
        new_attn[b, pl+n_abs:pl+n_abs+resp_len] = 1
    return new_data, new_attn  # prompt_len unchanged (ABS is response prefix)

# ── Build prefix layout and fill via recursion ────────────────────────────
N_ABS = K  # number of ABS placeholders per sequence
placeholder_tok = int(model.vocab_sizes[0].item())  # first abstract token ID

ed, ea = insert_prefix_abs(input_ids, attn_mask, prompt_len, N_ABS, placeholder_tok, pad_id)

with torch.no_grad():
    data, _, logits = model.recursion(
        ed, ea, max_iterations=2,
        memory_span_abs=1792, memory_span_traj=1792,
        temperature=1.0, prompt_len=prompt_len,
    )

print(f"=== Prefix-first: [Q] [ABS×{N_ABS}] [CoT] [#### ans]  (Jacobi-filled) ===\n")
for b in range(N_SAMPLES):
    valid = int(ea[b].sum())
    pl    = prompt_len[b].item()
    seq   = data[b, :valid]
    resp  = seq[pl:]

    # ABS prefix block = first N_ABS tokens of response
    abs_prefix = resp[:N_ABS]
    cot_plus   = resp[N_ABS:]

    ans_pos = (cot_plus == ANSWER_TOK).nonzero(as_tuple=True)[0]
    ai      = ans_pos[0].item() if len(ans_pos) else len(cot_plus)
    cot     = cot_plus[:ai]
    ans     = cot_plus[ai:]

    n_abs_pfx = (abs_prefix >= base_vocab).sum().item()
    n_nl_cot  = (cot < base_vocab).sum().item()
    n_abs_cot = (cot >= base_vocab).sum().item()

    print(f"[{b}]  total={valid}  |  prefix: {n_abs_pfx}/{N_ABS} ABS  |  CoT: {n_abs_cot} abs + {n_nl_cot} nl")
    print(f"  ABS prefix : {decode_annotated(abs_prefix)}")
    print(f"  CoT        : {decode_annotated(cot)[:180]}")
    print(f"  Ans        : {tokenizer.decode([t for t in ans.tolist() if t < base_vocab], skip_special_tokens=False).strip()[:60]}")
    print()

=== Prefix-first: [Q] [ABS×4] [CoT] [#### ans]  (Jacobi-filled) ===

[0]  total=104  |  prefix: 4/4 ABS  |  CoT: 0 abs + 54 nl
  ABS prefix : [ABS][ABS][ABS][ABS]
  CoT        :  Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.

  Ans        : #### 72

[1]  total=103  |  prefix: 4/4 ABS  |  CoT: 0 abs + 60 nl
  ABS prefix : [ABS][ABS][ABS][ABS]
  CoT        :  Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.

  Ans        : #### 10

[2]  total=168  |  prefix: 4/4 ABS  |  CoT: 0 abs + 97 nl
  ABS prefix : [ABS][ABS][ABS][ABS]
  CoT        :  In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 - 15 = $<<100-50-30-15=5>>
  Ans        : #### 5

[3]  total=187  |  prefix: 4/4 ABS  |  CoT: 0 abs + 122 nl
  ABS prefix : [ABS][ABS][ABS][ABS]
  CoT        :  Maila rea

In [34]:
# ── Kernel 2: NL Dropping on the prefix-first layout ───────────────────────
# Input  (from kernel 1, prefix-first):
#   [Q]  [ABS_1 ... ABS_N]  [nl_1 nl_2 ... nl_M]  [#### ans]
#
# _drop_nl_prefix_m_set samples m from M_SET, drops the first m NL tokens
# from the CoT region (everything between the ABS block and ####):
#
#   [Q]  [ABS_1 ... ABS_N]  [nl_{m+1} ... nl_M]  [#### ans]
#        ^^^^^^^^^^^^^^^^    ^^^^^^^^^^^^^^^^^^^
#         abs block intact    suffix NL only
#
# This is compatible with prefix-first because ABS tokens (>= base_vocab)
# are naturally skipped by the NL-position logic in _drop_nl_prefix_m_set.

import random
random.seed(42)
M_SET = (0, 16, 32, 64, 128)

# Use `data` and `ea` from Kernel 1 (prefix-first, Jacobi-filled)
print(f"M_SET = {M_SET}  (m sampled per-sequence independently)\n")

for trial in range(4):
    print(f"{'─'*72}  trial {trial + 1}")
    out_ids, out_attn, out_pl = _drop_nl_prefix_m_set(
        data, ea, prompt_len,
        base_vocab, pad_id, m_set=M_SET, answer_token_id=ANSWER_TOK,
    )
    for b in range(N_SAMPLES):
        valid = int(out_attn[b].sum())
        pl    = out_pl[b].item()
        resp  = out_ids[b, pl:valid]

        ans_pos = (resp == ANSWER_TOK).nonzero(as_tuple=True)[0]
        ai      = ans_pos[0].item() if len(ans_pos) else len(resp)
        cot     = resp[:ai]
        ans     = resp[ai:]

        n_abs = (cot >= base_vocab).sum().item()
        n_nl  = (cot <  base_vocab).sum().item()
        n_abs_ans = (ans >= base_vocab).sum().item()

        cot_text = decode_annotated(cot)
        ans_text = tokenizer.decode([t for t in ans.tolist() if t < base_vocab],
                                    skip_special_tokens=False)

        print(f"  [{b}]  {int(ea[b].sum())}→{valid}tok  |  CoT: {n_abs} abs + {n_nl} nl  |  ABS_ans={n_abs_ans}")
        print(f"       CoT: {cot_text[:180]}")
        print(f"       Ans: {ans_text.strip()[:50]}")
    print()

M_SET = (0, 16, 32, 64, 128)  (m sampled per-sequence independently)

────────────────────────────────────────────────────────────────────────  trial 1
  [0]  104→104tok  |  CoT: 4 abs + 54 nl  |  ABS_ans=0
       CoT: [ABS][ABS][ABS][ABS] Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.

       Ans: #### 72
  [1]  103→103tok  |  CoT: 4 abs + 60 nl  |  ABS_ans=0
       CoT: [ABS][ABS][ABS][ABS] Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.

       Ans: #### 10
  [2]  168→136tok  |  CoT: 4 abs + 65 nl  |  ABS_ans=0
       CoT: [ABS][ABS][ABS][ABS] grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 - 15 = $<<100-50-30-15=5>>5 more.

       Ans: #### 5
  [3]  187→171tok  |  CoT: 4 abs + 106 nl  |  ABS_ans=0
       CoT: [ABS][ABS][ABS][ABS]24>>24 pages today.
So she was able to read a total of 12 + 24 = <<12+24=36>>36 pages s

In [30]:
# ── Option 1: Free-form generation ──────────────────────────────────────────
# Training layout:  [Q] [ABS×N] [NL_CoT] [#### ans]   (cot_only_abs=True)
# Eval  (this cell): model.generate(free_form=True)
#   → no forced ABS positions; model generates ABS prefix + NL CoT naturally
#   → if the model has learned the prefix format, it produces [ABS×?][NL_CoT][####][ans]
#   → if not yet trained, it mostly produces NL (random init)
#
# This is Option 1: unconstrained — the model controls how many ABS tokens to emit.

QUERY_ONLY = input_ids  # just prompts from the GSM8K batch

with torch.no_grad():
    gen_ff = model.generate(
        QUERY_ONLY, max_new_tokens=120,
        attention_mask=attn_mask,
        temperature=0.0,
        free_form=True,         # ← Option 1: no forced positions
        K=None,
    )

print("=== Option 1: free_form=True  (model controls ABS output) ===\n")
for b in range(N_SAMPLES):
    pl   = prompt_len[b].item()
    resp = gen_ff[b, pl:]

    ans_pos = (resp == ANSWER_TOK).nonzero(as_tuple=True)[0]
    ai      = ans_pos[0].item() if len(ans_pos) else len(resp)
    cot     = resp[:ai]
    ans     = resp[ai:]

    n_abs = (cot >= base_vocab).sum().item()
    n_nl  = (cot <  base_vocab).sum().item()
    cot_text = decode_annotated(cot)
    ans_text = tokenizer.decode([t for t in ans.tolist() if t < base_vocab], skip_special_tokens=False)

    print(f"[{b}]  CoT: {n_abs} abs + {n_nl} nl")
    print(f"  {cot_text[:220]}")
    print(f"  Ans: {ans_text.strip()[:60]}")
    print()

=== Option 1: free_form=True  (model controls ABS output) ===

[0]  CoT: 0 abs + 54 nl
   Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.

  Ans: #### 72<|endoftext|><|endoftext|><|endoftext|><|endoftext|><

[1]  CoT: 0 abs + 60 nl
   Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.

  Ans: #### 10<|endoftext|><|endoftext|><|endoftext|><|endoftext|><

[2]  CoT: 0 abs + 97 nl
   In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 - 15 = $<<100-50-30-15=5>>5 more.

  Ans: #### 5<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|

[3]  CoT: 0 abs + 122 nl
   Maila read 12 x 2 = <<12*2=24>>24 pages today.
So she was able to read a total of 12 + 24 = <<12+24=36>>36 pages since yesterday.
There are 120 - 36 = <<120-36=84>>84 pages left to be read.
Since she wan

In [31]:
# ── Option 2: Fixed abs prefix (pause-token style) ───────────────────────────
# Training layout:  [Q] [ABS×8] [NL_CoT] [#### ans]   (cot_only_abs=True, abs_prefix_max=8)
# Eval  (this cell): model.generate(abs_prefix_max=8)
#   → Phase 1: model forced to generate exactly 8 ABS tokens (like pause tokens)
#   → Phase 2: model generates NL CoT + answer freely after the 8-token thinking budget
#
# This is Option 2: constrained budget — fixed 8 ABS "thinking" tokens regardless of sequence length.
# Compare with Option 1: here the budget is fixed, not model-controlled.

ABS_BUDGET = 8

with torch.no_grad():
    gen_prefix = model.generate(
        QUERY_ONLY, max_new_tokens=120,
        attention_mask=attn_mask,
        temperature=0.0,
        abs_prefix_max=ABS_BUDGET,   # ← Option 2: force exactly 8 ABS, then NL
    )

print(f"=== Option 2: abs_prefix_max={ABS_BUDGET}  (fixed ABS budget, then free NL) ===\n")
for b in range(N_SAMPLES):
    pl   = prompt_len[b].item()
    resp = gen_prefix[b, pl:]

    ans_pos = (resp == ANSWER_TOK).nonzero(as_tuple=True)[0]
    ai      = ans_pos[0].item() if len(ans_pos) else len(resp)
    cot     = resp[:ai]
    ans     = resp[ai:]

    n_abs = (cot >= base_vocab).sum().item()
    n_nl  = (cot <  base_vocab).sum().item()
    cot_text = decode_annotated(cot)
    ans_text = tokenizer.decode([t for t in ans.tolist() if t < base_vocab], skip_special_tokens=False)

    print(f"[{b}]  CoT: {n_abs} abs (forced={ABS_BUDGET}) + {n_nl} nl")
    print(f"  {cot_text[:220]}")
    print(f"  Ans: {ans_text.strip()[:60]}")
    print()

TypeError: SorlModelWrapper.generate() got an unexpected keyword argument 'abs_prefix_max'